### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

#### 1️⃣ 필요한 라이브러리 설치 및 불러오기

In [ ]:
# YOLO 라이브러리 설치 (최초 1회만 실행)
!pip install ultralytics

In [ ]:
# 필요한 라이브러리 불러오기
import os
import warnings
warnings.filterwarnings('ignore')

# 🎯 [미션] ultralytics 라이브러리에서 YOLO 모듈을 불러오세요.
from ultralytics import _______

print("✅ 라이브러리 로딩 완료!")

---
#### 2️⃣ 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/2026_AI_Advanced_Study-main/4차시/03_plant_disease/code"

In [ ]:
# 데이터 경로 설정
DATA_PATH = '../data'

print(f"✅ 데이터 경로: {DATA_PATH}")

---
#### 3️⃣ 학습된 모델 불러오기

01_train.ipynb에서 학습한 모델을 불러옵니다.  
학습 결과는 `runs/classify/train/weights/` 폴더에 저장됩니다.

In [ ]:
# 🎯 [미션] 학습된 모델의 경로를 입력하세요.
# 힌트: Classification 결과는 runs/classify/ 폴더에 저장됩니다.
MODEL_PATH = _______

# 모델 로드
model = YOLO(MODEL_PATH)

print(f"✅ 모델 로드 완료: {MODEL_PATH}")

---
#### 4️⃣ 테스트 데이터셋으로 모델 평가

학습된 모델의 성능을 테스트 데이터셋으로 평가합니다.

**Classification 평가 지표:**
- **Top-1 Accuracy**: 가장 높은 확률의 예측이 정답인 비율
- **Top-5 Accuracy**: 상위 5개 예측 중 정답이 포함된 비율

In [ ]:
print("📊 테스트 데이터셋으로 모델 평가 중...")

# 🎯 [미션] 모델을 검증하는 메서드를 호출하세요.
# 힌트: 학습은 train(), 검증은?
metrics = model._______(
    data=DATA_PATH,
    split='test'  # test 데이터셋 사용
)

print("\n🎉 평가 완료!")

In [ ]:
# 평가 결과 출력
print("=" * 50)
print("📊 Classification 성능 평가 결과")
print("=" * 50)
print(f"\n🎯 Top-1 Accuracy: {metrics.top1:.4f} ({metrics.top1*100:.2f}%)")
print(f"🎯 Top-5 Accuracy: {metrics.top5:.4f} ({metrics.top5*100:.2f}%)")
print("\n" + "=" * 50)

---
#### 5️⃣ 혼동 행렬 시각화

혼동 행렬(Confusion Matrix)을 통해 각 클래스별 예측 성능을 확인합니다.

In [ ]:
from IPython.display import Image, display

# 혼동 행렬 이미지 경로
confusion_img = "./runs/classify/train/confusion_matrix.png"

if os.path.exists(confusion_img):
    print("=== 혼동 행렬 (Confusion Matrix) ===")
    display(Image(filename=confusion_img, width=600))
else:
    print("혼동 행렬 이미지를 찾을 수 없습니다.")
    print("01_train.ipynb를 먼저 실행하세요.")

---
#### 6️⃣ 개별 이미지 예측 테스트

테스트 데이터셋에서 샘플 이미지를 선택하여 예측 결과를 확인합니다.

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image as PILImage

# 테스트 이미지 샘플 수집
test_images = []
test_path = os.path.join(DATA_PATH, 'test')

for cls_folder in os.listdir(test_path):
    cls_path = os.path.join(test_path, cls_folder)
    if os.path.isdir(cls_path):
        images = glob.glob(os.path.join(cls_path, '*.[jJ][pP][gG]'))[:1]  # 각 클래스당 1장
        test_images.extend(images)

print(f"📷 테스트할 이미지 수: {len(test_images)}장")

In [ ]:
# 각 이미지에 대해 예측 수행
fig, axes = plt.subplots(1, len(test_images), figsize=(4*len(test_images), 4))

if len(test_images) == 1:
    axes = [axes]

for idx, img_path in enumerate(test_images):
    # 예측 수행
    results = model.predict(img_path, verbose=False)
    
    # 결과 추출
    probs = results[0].probs
    predicted_class = results[0].names[probs.top1]
    confidence = probs.top1conf.item()
    
    # 실제 클래스 (폴더명에서 추출)
    actual_class = os.path.basename(os.path.dirname(img_path))
    
    # 이미지 표시
    img = PILImage.open(img_path)
    axes[idx].imshow(img)
    
    # 예측 결과 표시
    color = 'green' if predicted_class == actual_class else 'red'
    axes[idx].set_title(f"예측: {predicted_class}\n({confidence*100:.1f}%)\n실제: {actual_class}", 
                        color=color, fontsize=10)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()